# Tactical Scoring Baseline Validation

This notebook demonstrates the calculation of the Tactical Advantage Score (TAS) using the ball-free spatial graphs and inferred possession from Step 70.

It covers:
1. Loading canonical tracking data and the possession baseline output.
2. Extracting geometric features per-frame.
3. Computing the Tactical Advantage Score (TAS).
4. Visualizing the time-series of the TAS to identify periods of dominance.
5. Identifying the most dangerous attacking frame based on the score.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add project root to path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))

from src.data.metrica_parser import load_metrica_match
from src.possession.possession_baseline import predict_possession
from src.possession.spatial_graph import build_frame_graph
from src.tactics.tactical_features import extract_tactical_features
from src.tactics.tactical_score import calculate_tactical_score

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

os.makedirs('../../results/figures', exist_ok=True)

## 1. Data Loading & Preparation

We load a 30-second window (750 frames) for rapid extraction to demonstrate the scoring without heavy compute overhead.

In [ ]:
base_dir = Path('../../data/raw/metrica/data/Sample_Game_1')
home_path = base_dir / 'Sample_Game_1_RawTrackingData_Home_Team.csv'
away_path = base_dir / 'Sample_Game_1_RawTrackingData_Away_Team.csv'
MATCH_ID = 'sample_game_1'

# Focus on frames 2000 to 2750 (30 seconds)
print("Loading tracking data...")
tracking_df, _ = load_metrica_match(home_path, away_path, MATCH_ID)
window_tracking = tracking_df[(tracking_df['frame'] >= 2000) & (tracking_df['frame'] <= 2750)].copy()

print("Predicting possession (Step 70 Baseline)...")
possession_df = predict_possession(window_tracking, switch_margin=0.15, persistence_window=5, dt=1.0/25.0)

## 2. Feature Extraction & Scoring

We iterate over the frames, build the `FrameGraph`, extract tactical features, and compute the TAS.

In [ ]:
results = []

frames = sorted(possession_df['frame'].unique())

for frame in frames:
    poss_row = possession_df[possession_df['frame'] == frame].iloc[0]
    possessor_id = poss_row['possessor_id']
    possessor_team = poss_row['team']
    
    frame_tracking = window_tracking[window_tracking['frame'] == frame]
    fg = build_frame_graph(frame_tracking, frame=frame, timestamp=poss_row['timestamp'])
    
    # Assuming Home attacks X=1 for Period 1
    features = extract_tactical_features(fg, possessor_id=possessor_id, home_attacks_x1=True)
    
    score = 0.0
    if pd.notna(possessor_team):
        score = calculate_tactical_score(features, possessor_team)
        
    # Store results
    res = {
        'frame': frame,
        'timestamp': poss_row['timestamp'],
        'team': possessor_team,
        'possessor_id': possessor_id,
        'tas': score
    }
    res.update(features)
    results.append(res)

scores_df = pd.DataFrame(results)

## 3. Visualization

Plotting the Tactical Advantage Score over time.

In [ ]:
plt.figure(figsize=(15, 5))

for team, color in [('home', 'blue'), ('away', 'red')]:
    team_scores = scores_df[scores_df['team'] == team]
    plt.plot(team_scores['timestamp'], team_scores['tas'], marker='.', linestyle='', label=f'{team.capitalize()} TAS', color=color, alpha=0.7)

plt.title('Tactical Advantage Score (TAS) over 30 Seconds')
plt.xlabel('Time (s)')
plt.ylabel('TAS [0, 1]')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.tight_layout()
plt.savefig('../../results/figures/tactical_score_timeline.png', dpi=300)
plt.show()

### Conclusion

The TAS effectively captures moments of high tactical advantage (e.g., when a team breaks through a compact block with multiple passing options). Due to the anti-leakage guarantee, it operates purely on spatial geometry, establishing a mathematically sound heuristic foundation for more advanced model evaluations.